In [1]:
import os

# ------------- MUST be set before importing datasets/transformers -------------
os.environ["HF_HOME"] = "/kaggle/temp/hf"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/temp/hf/datasets"
os.environ["HF_HUB_CACHE"] = "/kaggle/temp/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/temp/hf/transformers"
os.environ["XDG_CACHE_HOME"] = "/kaggle/temp/.cache"

# Create dirs
for p in [
    os.environ["HF_HOME"],
    os.environ["HF_DATASETS_CACHE"],
    os.environ["HF_HUB_CACHE"],
    os.environ["TRANSFORMERS_CACHE"],
    os.environ["XDG_CACHE_HOME"],
]:
    os.makedirs(p, exist_ok=True)

print("HF_HOME =", os.environ["HF_HOME"])
print("HF_DATASETS_CACHE =", os.environ["HF_DATASETS_CACHE"])
print("HF_HUB_CACHE =", os.environ["HF_HUB_CACHE"])
print("TRANSFORMERS_CACHE =", os.environ["TRANSFORMERS_CACHE"])
print("XDG_CACHE_HOME =", os.environ["XDG_CACHE_HOME"])

HF_HOME = /kaggle/temp/hf
HF_DATASETS_CACHE = /kaggle/temp/hf/datasets
HF_HUB_CACHE = /kaggle/temp/hf/hub
TRANSFORMERS_CACHE = /kaggle/temp/hf/transformers
XDG_CACHE_HOME = /kaggle/temp/.cache


In [2]:
!du -sh /kaggle/working/* 2>/dev/null || true
!rm -rf /kaggle/working/.cache
!rm -rf /kaggle/working/hf
!rm -rf /kaggle/working/huggingface
!du -sh /kaggle/working/* 2>/dev/null || true

28K	/kaggle/working/__notebook__.ipynb
28K	/kaggle/working/__notebook__.ipynb


In [3]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("abnerh/TORGO-database")

# Convert to pandas from Arrow (this preserves audio.path as a dict-like)
df = ds["train"].to_pandas()

# audio column is a dict with keys: bytes, path
df["audio_path"] = df["audio"].apply(lambda x: x["path"])
df["speaker"] = df["audio_path"].apply(lambda p: str(p).split("_")[0])

print(df[["audio_path", "speaker", "speech_status", "transcription"]].head())
print("Unique speakers:", sorted(df["speaker"].unique()))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/370M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/397M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/356M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/441M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16552 [00:00<?, ? examples/s]

                 audio_path speaker speech_status  \
0  FC01_1_arrayMic_0066.wav    FC01       healthy   
1  FC01_1_arrayMic_0016.wav    FC01       healthy   
2  FC01_1_arrayMic_0061.wav    FC01       healthy   
3  FC01_1_arrayMic_0053.wav    FC01       healthy   
4  FC01_1_arrayMic_0043.wav    FC01       healthy   

                                       transcription  
0                                              alpha  
1                                                the  
2  Except in the winter when the ooze or snow or ...  
3                                               raid  
4                                              read   
Unique speakers: ['F01', 'F03', 'F04', 'FC01', 'FC02', 'FC03', 'M01', 'M02', 'M03', 'M04', 'M05', 'MC01', 'MC02', 'MC03', 'MC04']


In [4]:
severity_mapping = {
    # Dysarthric
    "F01":"severe","M01":"severe","M02":"severe","M04":"severe",
    "M05":"moderate","F03":"moderate",
    "F04":"mild","M03":"mild",

    # Control
    "FC01":"control","FC02":"control","FC03":"control",
    "MC01":"control","MC02":"control","MC03":"control","MC04":"control"
}

df["Category"] = df["speaker"].map(severity_mapping)

# sanity check
print(df["Category"].value_counts(dropna=False))

unmapped = sorted(df[df["Category"].isna()]["speaker"].unique())
print("Unmapped speakers:", unmapped)

Category
control     10978
severe       2417
moderate     1674
mild         1483
Name: count, dtype: int64
Unmapped speakers: []


In [5]:
import pandas as pd

# count utterances per speaker
spk_counts = df.groupby(["speaker","Category"]).size().reset_index(name="n")

# for each category, pick the speaker with the smallest n
test_spk = set(
    spk_counts.sort_values("n").groupby("Category").first().reset_index()["speaker"].tolist()
)

train_spk = set(df["speaker"].unique()) - test_spk

df["split"] = df["speaker"].apply(lambda s: "test" if s in test_spk else "train")

print("Test speakers:", sorted(test_spk))
print(pd.crosstab(df["split"], df["Category"]))

Test speakers: ['F01', 'F04', 'FC01', 'M05']
Category  control  mild  moderate  severe
split                                    
test          302   673       587     236
train       10676   810      1087    2181


In [6]:
from datasets import Dataset, DatasetDict

# Recreate full HF dataset with metadata
hf_full = ds["train"].add_column("speaker", df["speaker"].tolist())
hf_full = hf_full.add_column("Category", df["Category"].tolist())
hf_full = hf_full.add_column("split", df["split"].tolist())

# Create DatasetDict
hf_dict = DatasetDict({
    "train": hf_full.filter(lambda x: x["split"] == "train"),
    "test":  hf_full.filter(lambda x: x["split"] == "test"),
})

SAVE_HF_PATH = "/kaggle/working/torgo_hf_speaker_split"

hf_dict.save_to_disk(SAVE_HF_PATH)

print("Saved HF dataset to:", SAVE_HF_PATH)

print(hf_dict)

Filter:   0%|          | 0/16552 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16552 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/14754 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1798 [00:00<?, ? examples/s]

Saved HF dataset to: /kaggle/working/torgo_hf_speaker_split
DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'speech_status', 'gender', 'duration', 'speaker', 'Category', 'split'],
        num_rows: 14754
    })
    test: Dataset({
        features: ['audio', 'transcription', 'speech_status', 'gender', 'duration', 'speaker', 'Category', 'split'],
        num_rows: 1798
    })
})


In [7]:
# ---------------------------
# 1) Install deps
# ---------------------------
!pip -q install -U transformers datasets accelerate evaluate jiwer soundfile packaging

# ---------------------------
# 2) Imports (AFTER env vars are set)
# ---------------------------
import numpy as np
import torch
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_from_disk
import transformers
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
)

print("transformers:", transformers.__version__)

# ---------------------------
# 3) Optional cleanup of old output-side cache
#    Safe to run if you want to keep /kaggle/working small
# ---------------------------
!rm -rf /kaggle/working/.cache
!rm -rf /kaggle/working/hf
!rm -rf /kaggle/working/huggingface

# ---------------------------
# 4) Load your saved speaker-split dataset
# ---------------------------
DATASET_PATH = "/kaggle/working/torgo_hf_speaker_split"  # change if needed
hf_dict = load_from_disk(DATASET_PATH)

train_ds = hf_dict["train"]
test_ds  = hf_dict["test"]

AUDIO_COL = "audio"
TEXT_COL  = "transcription"
CAT_COL   = "Category"

print("Train:", train_ds)
print("Test :", test_ds)
print("Train columns:", train_ds.column_names)
print("Test columns :", test_ds.column_names)

assert AUDIO_COL in train_ds.column_names, f"Missing {AUDIO_COL}"
assert TEXT_COL  in train_ds.column_names, f"Missing {TEXT_COL}"
assert CAT_COL   in test_ds.column_names,  f"Missing {CAT_COL} in test set"

# ---------------------------
# 5) Load Whisper Small + processor
#    Cache model files in /kaggle/temp too
# ---------------------------
MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    cache_dir=os.environ["TRANSFORMERS_CACHE"],
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    cache_dir=os.environ["TRANSFORMERS_CACHE"],
)

# IMPORTANT: generation settings should live in generation_config, not model.config
if hasattr(model.config, "suppress_tokens"):
    model.config.suppress_tokens = None
if hasattr(model.config, "forced_decoder_ids"):
    model.config.forced_decoder_ids = None

model.generation_config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None

# ---------------------------
# 6) Preprocess: audio -> input_features, text -> labels
# ---------------------------
def prepare_batch(batch):
    audio = batch[AUDIO_COL]
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(
        batch[TEXT_COL],
        truncation=True
    ).input_ids

    return batch

# Keep Category in test set for later by-class eval
remove_train_cols = train_ds.column_names
remove_test_cols  = [c for c in test_ds.column_names if c != CAT_COL]

# Force map cache files into /kaggle/temp so they DO NOT count against Output quota
train_cache_file = "/kaggle/temp/hf/train_preprocessed.arrow"
test_cache_file  = "/kaggle/temp/hf/test_preprocessed.arrow"

train_p = train_ds.map(
    prepare_batch,
    remove_columns=remove_train_cols,
    cache_file_name=train_cache_file,
    load_from_cache_file=True,
    desc="Preprocessing train",
)

test_p = test_ds.map(
    prepare_batch,
    remove_columns=remove_test_cols,
    cache_file_name=test_cache_file,
    load_from_cache_file=True,
    desc="Preprocessing test",
)

print("Processed train:", train_p)
print("Processed test :", test_p)
print("Processed keys train:", train_p.column_names)
print("Processed keys test :", test_p.column_names)

print("Train cache file exists:", os.path.exists(train_cache_file))
print("Test cache file exists :", os.path.exists(test_cache_file))

# ---------------------------
# 7) Data collator
# ---------------------------
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        if CAT_COL in features[0]:
            batch[CAT_COL] = [f.get(CAT_COL, None) for f in features]

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# ---------------------------
# 8) WER metric
# ---------------------------
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

    label_ids = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    return {
        "wer": wer_metric.compute(predictions=pred_str, references=label_str)
    }

# ---------------------------
# 9) Loss logging callback
# ---------------------------
class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        if "loss" in logs:
            print(f"[step {state.global_step}] train_loss={logs['loss']:.4f}")
        if "learning_rate" in logs:
            print(f"[step {state.global_step}] lr={logs['learning_rate']:.2e}")

# ---------------------------
# 10) Sanity check disk usage
# ---------------------------
print("\n--- DISK USAGE CHECK ---")
!du -sh /kaggle/working/* 2>/dev/null || true
!du -sh /kaggle/temp/* 2>/dev/null || true

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

generation_config.json: 0.00B [00:00, ?B/s]

Preprocessing train:   0%|          | 0/14754 [00:00<?, ? examples/s]

Preprocessing test:   0%|          | 0/1798 [00:00<?, ? examples/s]

Processed train: Dataset({
    features: ['input_features', 'labels'],
    num_rows: 14754
})
Processed test : Dataset({
    features: ['Category', 'input_features', 'labels'],
    num_rows: 1798
})
Processed keys train: ['input_features', 'labels']
Processed keys test : ['Category', 'input_features', 'labels']
Train cache file exists: True
Test cache file exists : True



--- DISK USAGE CHECK ---
52K	/kaggle/working/__notebook__.ipynb
1.5G	/kaggle/working/torgo_hf_speaker_split
19G	/kaggle/temp/hf


In [8]:
# ---------------------------
# 7) Training args (space-safe)
#    IMPORTANT:
#    - We DO NOT pass evaluation_strategy/eval_strategy (version-safe)
#    - We save rarely, and keep only 1 checkpoint
# ---------------------------

# Fix generation params being incorrectly stored in config
if hasattr(model.config, "suppress_tokens"):
    model.config.suppress_tokens = None

# Put generation params where they belong
model.generation_config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None

# Also do this for Whisper specifically (common gotcha)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = None

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/whisper_small_torgo_baseline",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=200,
    num_train_epochs=3,
    logging_steps=50,

    save_steps=999999,     # <--- effectively NO checkpoints
    save_total_limit=1,

    predict_with_generate=True,
    generation_max_length=225,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_p,
    eval_dataset=None,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[PrintLossCallback()],
)

# ---------------------------
# 8) Train
# ---------------------------
trainer.train()

# run the fix block above, then:
FINAL_DIR = "/kaggle/working/whisper_small_torgo_final"
trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print("Saved to:", FINAL_DIR)

Step,Training Loss
50,20.504880
100,7.965953
150,11.052395
200,12.803474
250,5.326647
300,3.255275
350,1.742969
400,0.648311
450,0.574020
500,0.412067


[step 50] train_loss=20.5049
[step 50] lr=2.45e-06
[step 100] train_loss=7.9660
[step 100] lr=4.95e-06
[step 150] train_loss=11.0524
[step 150] lr=7.45e-06
[step 200] train_loss=12.8035
[step 200] lr=9.95e-06
[step 250] train_loss=5.3266
[step 250] lr=9.91e-06
[step 300] train_loss=3.2553
[step 300] lr=9.81e-06
[step 350] train_loss=1.7430
[step 350] lr=9.72e-06
[step 400] train_loss=0.6483
[step 400] lr=9.63e-06
[step 450] train_loss=0.5740
[step 450] lr=9.53e-06
[step 500] train_loss=0.4121
[step 500] lr=9.44e-06
[step 550] train_loss=0.4024
[step 550] lr=9.35e-06
[step 600] train_loss=0.3420
[step 600] lr=9.25e-06
[step 650] train_loss=0.3508
[step 650] lr=9.16e-06
[step 700] train_loss=0.3181
[step 700] lr=9.06e-06
[step 750] train_loss=0.3071
[step 750] lr=8.97e-06
[step 800] train_loss=0.2602
[step 800] lr=8.88e-06
[step 850] train_loss=0.2991
[step 850] lr=8.78e-06
[step 900] train_loss=0.2609
[step 900] lr=8.69e-06
[step 950] train_loss=0.2676
[step 950] lr=8.60e-06
[step 1000]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /kaggle/working/whisper_small_torgo_final


In [9]:
!pip install -q evaluate jiwer

# ============================================================
# SAVE WHISPER PREDICTIONS + COMPUTE WER/CER (overall + by class)
# Assumes you already have:
#   - trainer (trained)
#   - processor
#   - test_p (processed test dataset used for predict())
#   - test_ds (raw test dataset with metadata + transcription)
#   - TEXT_COL = "transcription"
#   - CAT_COL  = "Category"
# Optional (if present in test_ds): "speaker", "audio_path"
# Outputs:
#   /kaggle/working/torgo_whisper_small_predictions.csv
#   /kaggle/working/torgo_whisper_small_predictions.parquet
#   /kaggle/working/torgo_whisper_small_predictions_hf/   (HF dataset)
# ============================================================

import numpy as np
import pandas as pd
import evaluate
from collections import defaultdict
from jiwer import wer as wer_utt_fn, cer as cer_utt_fn
from datasets import Dataset

# ---------------------------
# 1) Metrics objects
# ---------------------------
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

# ---------------------------
# 2) Predict on test
# ---------------------------
pred = trainer.predict(test_p)

pred_ids = pred.predictions
if isinstance(pred_ids, tuple):  # some versions return tuple
    pred_ids = pred_ids[0]

# If pred_ids are logits, argmax them (rare, but safe)
if hasattr(pred_ids, "ndim") and pred_ids.ndim == 3:
    pred_ids = np.argmax(pred_ids, axis=-1)

pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

# ---------------------------
# 3) References + category labels (aligned to test_ds order)
# ---------------------------
TEXT_COL = "transcription"
CAT_COL  = "Category"

refs = test_ds[TEXT_COL]
cats = test_ds[CAT_COL]

assert len(pred_str) == len(refs) == len(cats), "Prediction and reference lengths do not match!"

# ---------------------------
# 4) Overall WER/CER
# ---------------------------
overall_wer = wer_metric.compute(predictions=pred_str, references=refs)
overall_cer = cer_metric.compute(predictions=pred_str, references=refs)

print("\n====================")
print(f"TEST overall WER: {overall_wer:.4f}")
print(f"TEST overall CER: {overall_cer:.4f}")
print("====================\n")

# ---------------------------
# 5) WER/CER by Category
# ---------------------------
by_cat_preds = defaultdict(list)
by_cat_refs  = defaultdict(list)

for p, r, c in zip(pred_str, refs, cats):
    by_cat_preds[c].append(p)
    by_cat_refs[c].append(r)

print("WER/CER by Category:")
by_cat_scores = {}
for c in sorted(by_cat_preds.keys()):
    c_wer = wer_metric.compute(predictions=by_cat_preds[c], references=by_cat_refs[c])
    c_cer = cer_metric.compute(predictions=by_cat_preds[c], references=by_cat_refs[c])
    by_cat_scores[c] = {"wer": c_wer, "cer": c_cer, "n": len(by_cat_preds[c])}
    print(f"{c:>9s}: WER={c_wer:.4f}  CER={c_cer:.4f}  (n={len(by_cat_preds[c])})")

# ---------------------------
# 6) Build a predictions "dataset" (DataFrame)
#    Includes optional metadata if available
# ---------------------------
cols_available = set(test_ds.column_names)

pred_df = pd.DataFrame({
    "Category": cats,
    "ground_truth": refs,
    "prediction": pred_str,
})

# Optional metadata
if "speaker" in cols_available:
    pred_df["speaker"] = test_ds["speaker"]

if "audio_path" in cols_available:
    pred_df["audio_path"] = test_ds["audio_path"]
elif "audio" in cols_available:
    # try to extract audio path if audio is dict-like
    try:
        aud = test_ds["audio"]
        pred_df["audio_path"] = [a.get("path") if isinstance(a, dict) else None for a in aud]
    except Exception:
        pred_df["audio_path"] = None

if "duration" in cols_available:
    pred_df["duration"] = test_ds["duration"]

# Per-utterance WER/CER (use jiwer; can take a bit but fine for TORGO)
pred_df["wer_utt"] = [wer_utt_fn(r, p) for r, p in zip(pred_df["ground_truth"], pred_df["prediction"])]
pred_df["cer_utt"] = [cer_utt_fn(r, p) for r, p in zip(pred_df["ground_truth"], pred_df["prediction"])]

# Add overall metrics as constant columns (handy when sharing file)
pred_df["overall_wer"] = overall_wer
pred_df["overall_cer"] = overall_cer

# ---------------------------
# 7) Save to disk (CSV + Parquet)
# ---------------------------
CSV_PATH = "/kaggle/working/torgo_whisper_small_predictions.csv"
PARQUET_PATH = "/kaggle/working/torgo_whisper_small_predictions.parquet"

pred_df.to_csv(CSV_PATH, index=False)
pred_df.to_parquet(PARQUET_PATH, index=False)

print("\nSaved prediction files:")
print("CSV    :", CSV_PATH)
print("Parquet:", PARQUET_PATH)

# ---------------------------
# 9) Optional: quick qualitative samples
# ---------------------------
print("\n===== SAMPLE PREDICTIONS =====\n")
num_samples = 12
num_samples = min(num_samples, len(pred_df))
idxs = np.random.choice(len(pred_df), size=num_samples, replace=False)

for i in idxs:
    print(f"[Category: {pred_df.loc[i,'Category']}]  WER={pred_df.loc[i,'wer_utt']:.3f}  CER={pred_df.loc[i,'cer_utt']:.3f}")
    if "speaker" in pred_df.columns:
        print("Speaker:", pred_df.loc[i, "speaker"])
    if "audio_path" in pred_df.columns:
        print("Audio :", pred_df.loc[i, "audio_path"])
    print("GT  :", pred_df.loc[i, "ground_truth"])
    print("PRED:", pred_df.loc[i, "prediction"])
    print("-" * 70)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA


TEST overall WER: 0.1205
TEST overall CER: 0.0751

WER/CER by Category:
  control: WER=0.0338  CER=0.0156  (n=302)
     mild: WER=0.0287  CER=0.0190  (n=673)
 moderate: WER=0.1873  CER=0.1188  (n=587)
   severe: WER=0.3333  CER=0.2126  (n=236)

Saved prediction files:
CSV    : /kaggle/working/torgo_whisper_small_predictions.csv
Parquet: /kaggle/working/torgo_whisper_small_predictions.parquet

===== SAMPLE PREDICTIONS =====

[Category: mild]  WER=0.000  CER=0.000
Speaker: F04
Audio : None
GT  : Please open the window quickly
PRED: Please open the window quickly
----------------------------------------------------------------------
[Category: control]  WER=1.000  CER=0.500
Speaker: FC01
Audio : None
GT  : feet
PRED: seat
----------------------------------------------------------------------
[Category: mild]  WER=0.000  CER=0.000
Speaker: F04
Audio : None
GT  : Did dad do academic bidding
PRED: Did dad do academic bidding
------------------------------------------------------------------